In [0]:
CREATE CATALOG km;
CREATE SCHEMA IF NOT EXISTS km.retail_demo;
CREATE VOLUME IF NOT EXISTS km.retail_demo.retail_files;

In [0]:
%python
from pyspark.sql import functions as F

sample_data = [
    (1001, "2026-10-25", "Chennai",   "Laptop",  2, 50000.0),
    (1002, "2026-10-25", "Mumbai",    "Mouse",   3,   500.0),
    (1003, "2026-10-25", "Chennai",   "Keyboard",2,  1500.0),
    (1004, "2026-10-26", "Bengaluru", "Laptop",  1, 50000.0),
    (1005, "2026-10-26", "Delhi",     "Mouse",   5,   500.0),
    (1006, "2026-10-26", "Mumbai",    "Monitor", 2, 12000.0),
    (1007, "2026-10-26", "Chennai",   "Mouse",  -1,   500.0),
    (1008, "2026-10-27", "Delhi",     "Keyboard",None, 1500.0)
]

columns = [
    "order_id", "order_date", "city",
    "product", "quantity", "unit_price"
]

source_df = spark.createDataFrame(sample_data, columns)

source_path = (
    "/Volumes/km/retail_demo/retail_files/incoming/sales"
)

(source_df.write
    .mode("overwrite")
    .option("header", True)
    .csv(source_path))

display(source_df)

In [0]:
%python
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(source_path)
)

(raw_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("km.retail_demo.bronze_sales"))

display(
    spark.table("km.retail_demo.bronze_sales")
)

In [0]:
%python
typed_df = (
    spark.table("km.retail_demo.bronze_sales")
    .withColumn("order_id", F.col("order_id").cast("long"))
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn(
        "total_amount",
        F.col("quantity") * F.col("unit_price")
    )
)

In [0]:
%python
valid_condition = (
    F.col("order_id").isNotNull()
    & F.col("order_date").isNotNull()
    & F.col("city").isNotNull()
    & F.col("product").isNotNull()
    & F.col("quantity").isNotNull()
    & (F.col("quantity") > 0)
    & F.col("unit_price").isNotNull()
    & (F.col("unit_price") >= 0)
)

silver_df = typed_df.filter(valid_condition)

invalid_df = typed_df.filter(~valid_condition)

In [0]:
%python
(silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("km.retail_demo.silver_sales"))

(invalid_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("km.retail_demo.quarantine_sales"))

display(silver_df)
display(invalid_df)

In [0]:
%python
silver_df = spark.table("km.retail_demo.silver_sales")

gold_df = (
    silver_df
    .groupBy("order_date", "city", "product")
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.countDistinct("order_id").alias("order_count"),
        F.sum("quantity").alias("total_quantity")
    )
)

(gold_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("km.retail_demo.gold_daily_sales"))

display(
    spark.table("km.retail_demo.gold_daily_sales")
    .orderBy("order_date", "city", "product")
)

In [0]:
%python
bronze_count = (
    spark.table("km.retail_demo.bronze_sales").count()
)

silver_count = (
    spark.table("km.retail_demo.silver_sales").count()
)

quarantine_count = (
    spark.table("km.retail_demo.quarantine_sales").count()
)

gold_sales = (
    spark.table("km.retail_demo.gold_daily_sales")
    .agg(F.sum("total_sales").alias("sales"))
    .first()["sales"]
)

print("Bronze count:", bronze_count)
print("Silver count:", silver_count)
print("Quarantine count:", quarantine_count)
print("Gold sales total:", gold_sales)

assert bronze_count == silver_count + quarantine_count
assert gold_sales == 181000.0